# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. {a: 4, b: 2} -> [4, 2]
2. [1,1] [2,1]
[1,3] None(NaN) не считается
[2,1] считается кол-во строк по группе с
3. 20
потому что по дефолту считается среднее. [30, 10] по ярлыку а 40/2 = 20
4. 2 3
[1, 2] потому что берем inner. то есть только то что сджойнилось
[1, 2, 3] поточу что берем left. то есть то что сджойнилось + то что есть в a
5. 4
Потому что в таблице b есть дубликат по id. Соответственно произошло задубление при джойне

In [91]:
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))

[4, 2]
[1, 1] [2, 1]
20.0
2 3
4


In [92]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [93]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [94]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [95]:
# 1
by_city = df.groupby("city").agg(orders=("order_id", "count"),
                                 cancel_share=("cancel", "mean"),
                                 avg_days=("delivery_days", "mean"))

by_city["revenue"] = paid.groupby("city")["amount"].sum()
by_city.sort_values("revenue", ascending=False).round(3)

,orders,cancel_share,avg_days,revenue
city,,,,
Алматы,2116,0.078,1.997,29144090.0
Астана,1339,0.088,2.019,17884750.0
Шымкент,776,0.206,4.451,8602660.0
Караганда,519,0.091,1.941,7020200.0
Актобе,387,0.106,2.070,5117380.0
Атырау,143,0.077,2.036,1893850.0


Алматы выделяется большим кол-вом заказов, быстрой доставкой, большой выручкой

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [96]:
# 2
paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean").round(3)


month,9,10
category,,
Дом,10499.433,10768.466
Книги,4020.000,3815.753
Красота,6187.085,6158.598
Одежда,13520.796,13045.598
Электроника,32772.750,33600.198


Было 8 стало 1. С этими функциями намного проще.

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [97]:
# 3 
df.groupby("delivery_days").agg(orders=("order_id", "count"),
                                cancel_share=("cancel", "mean"))

,orders,cancel_share
delivery_days,,
1.0,1557,0.042389
2.0,1570,0.079618
3.0,1590,0.133962
5.0,86,0.220930
6.0,120,0.216667
7.0,95,0.347368
8.0,104,0.432692


На 40% больше, это значит нужно срочно поработать над скоростью заказа.

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] - p[9]
print(p["chg"].round(3))
```

In [98]:
# 4
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["chg"] = p[10] - p[9]
print(p["chg"].round(3))
p["prc"] = p["chg"] / p[9] * 100
print(p["prc"].round(3))

city
Актобе       -36540.0
Алматы       141950.0
Астана      -183690.0
Атырау        45870.0
Караганда   -276840.0
Шымкент     -514640.0
Name: chg, dtype: float64
city
Актобе       -1.418
Алматы        0.979
Астана       -2.033
Атырау        4.964
Караганда    -7.588
Шымкент     -11.289
Name: prc, dtype: float64


Абсолютное изменение -514640, относительное изменение -11.289%

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [99]:
# 5
customers = pd.read_csv("customers.csv")
m = df.merge(customers, on="customer_id", how="left")
print(len(m) == len(df)) # len OK
m["age_group"] = m["age"] // 10 * 10
m_group = m.groupby("age_group").agg(orders=("order_id", "count"),
                           cancel=("cancel", "mean"))

m_group['mean_amt'] = m[m['status'] == 'оплачен'].groupby("age_group")["amount"].mean()
m_group['median_amt'] = m[m['status'] == 'оплачен'].groupby("age_group")["amount"].median()

m_group

True


,orders,cancel,mean_amt,median_amt
age_group,,,,
10,229,0.087336,14044.736842,10700.0
20,1226,0.096248,14130.415162,10100.0
30,1241,0.111201,15142.339075,10670.0
40,1170,0.111111,14792.971154,10050.0
50,1294,0.091963,14671.795745,10240.0
60,120,0.133333,16779.134615,10495.0


Какие возрасты попали в группу 60 и сколько в ней заказов?

Туда попали клиенты от 60 до 69 включительно. Всего заказов 120.

Подтверждает ли медиана вывод AI? 

Нет медиана показывает, что 60 занимают топ-3 по медианной выручке. По отменам смотря с какой точки зрения смотреть. Процент действительно больщшой, но он нивелируется малым количеством заказов.

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [100]:
# 6
customer_cnt = m.groupby("customer_id").size()
print('Сколько заказов у каждого клиента?', end='\n')
print(customer_cnt)

order_cnt = customer_cnt.value_counts().sort_index()
print('Сколько клиентов сделали 1, 2, 3… заказа?', end='\n')
print(order_cnt)

print('Какая доля клиентов купила больше одного раза?', end=' ')
print(round(order_cnt[1:10].sum() / order_cnt.sum() * 100, 2))


Сколько заказов у каждого клиента?
customer_id
C00001    4
C00002    1
C00003    2
C00004    2
C00005    4
         ..
C02196    1
C02197    3
C02198    1
C02199    2
C02200    1
Length: 2002, dtype: int64
Сколько клиентов сделали 1, 2, 3… заказа?
1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64
Какая доля клиентов купила больше одного раза? 74.98


In [101]:
cust_x_months = m.groupby("customer_id")["month"].nunique()
print('Сколько клиентов заказывали и в сентябре, и в октябре?', end=' ')
print(cust_x_months[cust_x_months == 2].sum())

Сколько клиентов заказывали и в сентябре, и в октябре? 2128


In [102]:
left_join = cu.merge(df, on="customer_id", how="left")
print('Сколько клиентов из customers.csv не заказали ничего?', end=' ')
print(len(left_join[left_join['order_id'].isna()]['customer_id']))

Сколько клиентов из customers.csv не заказали ничего? 198


Соединил через left join и это дало НаН в поле order_id. В логике работают несколько моментов.
1. В справочнике customers.csv customer_id уникальны
2. При соединении с заказами через left(при условии что слева ставим customers), если нет заказов то строка так и остается уникальной.
3. Чтобы вычислить клиентов без заказов проверяем через поле с номером заказа, так как оно не имеет пропусков в таблице с заказами.

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [103]:
# 7

# кол-во заказов за октябрь
# средний чек по оплаченным за октябрь
# доля отмен за сентябрь

# кол-во заказов за октябрь * (1 - доля отмен за сентябрь) * средний чек по оплаченным за октябрь

len_shym = len(df[(df['city'] == 'Шымкент') & (df['month'] == 10)])
mean_shym = paid[(paid['city'] == 'Шымкент') & (paid['month'] == 10)]['amount'].mean()
cancel_shym_9 = df[(df['city'] == 'Шымкент') & (df['month'] == 9)]['cancel'].mean()
cancel_shym_10 = df[(df['city'] == 'Шымкент') & (df['month'] == 10)]['cancel'].mean()

amt_shym_hyp = round(len_shym * (1 - cancel_shym_9) * mean_shym, 2)
amt_shym_fact = round(len_shym * (1 - cancel_shym_10) * mean_shym, 2)
loss = round(amt_shym_hyp - amt_shym_fact, 2)
compare = loss / float(823890)
diff = loss - float(823890)


print(f'Какой была бы выручка Шымкента в октябре? {amt_shym_hyp}')
print(f'Сколько компания потеряла из-за отмен? {loss}')
print(f'Сравните потерю с падением всей компании (823 890 ₸). Выходит разница в {compare:.1%}')
print(f'Что было бы с выручкой компании без сбоя? {diff}')

Какой была бы выручка Шымкента в октябре? 5315005.73
Сколько компания потеряла из-за отмен? 1270995.73
Сравните потерю с падением всей компании (823 890 ₸). Выходит разница в 154.3%
Что было бы с выручкой компании без сбоя? 447105.73


## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [104]:

results = []
rng = np.random.default_rng(7)

for i in range(100):
    probs = customers['channel'].value_counts(normalize=True)
    # реклама     0.515000
    # органика    0.337727
    # рефералы    0.147273

    group_codes = rng.choice(
        a=[0, 1, 2],
        size=len(customers),
        p=probs.values
    )

    group_names = {0: 'Group_A', 1: 'Group_B', 2: 'Group_C'}
    customers['group'] = pd.Series(group_codes).map(group_names)

    # Проверка итоговых пропорций
    #print(customers['group'].value_counts(normalize=True))

    customers_x_orders = paid.merge(customers, on="customer_id", how="left")
    cu_by_months = customers_x_orders.groupby(["group", "month"])["amount"].sum().unstack()
    cu_by_months["chg"] = (cu_by_months[10] - cu_by_months[9]) / cu_by_months[9] * 100
    #print(cu_by_months)
    results.append(cu_by_months.loc['Group_B', 'chg'])

In [105]:
results = pd.Series(results)
prc_5 = results.quantile(0.05)
prc_95 = results.quantile(0.95)
print(prc_5, prc_95)
print(prc_5 < -6.0 < prc_95)

-12.433072870582286 3.6734542403485526
True


-6.0% лежит внутри доверительного интервала случайных колебаний, что значит что это шум и ему нельзя доверять.